# Build a Multi-Agent System

**Prerequisites:** `04_orchestration_patterns.ipynb`

One system, out of everything this chapter built: a roster of specialists (nb1), a supervisor that routes to them (nb2), forms in both directions (nb3), a board they work on and an event that fires from it (nb3, nb4).

Then the half that decides whether any of it survives contact with a real day: **what happens when a worker fails, loops, or hands back something that cannot be used.** Every failure in this notebook is one this chapter already produced by accident.

| Where it happened | What went wrong |
|---|---|
| nb2, report B | a specialist ran out of budget and returned *"Stopping: reached the maximum of 8 iterations"*. The supervisor read it as an answer |
| nb2, dynamic delegation | the supervisor asked about "this serial" without the serial, and spent a whole agent run finding out |
| nb3, the board | two agents printed `record(...)` as text instead of calling the tool |
| nb4, the watcher | a condition tested the text for "7731" and missed a risk recorded as `no` |
| nb1, nb2 | answers marked internal, or confident with nothing behind them, sent outward anyway |

**By the end you'll have:**
- One recall desk built from the chapter's parts, handling a real report end to end
- A worker that raises, one that runs out of budget, and one that will not fill in the form — each arriving as a **form the caller can read** rather than as an exception or a sentence
- A gate that decides whether a reply may leave the building, and what it refuses
- A board whose watchers loop, and the guard that stops it
- An honest list of what this system still gets wrong

## Setup

## One system, five patterns

Notebook 4 ended on hybrids: not a category, just one choice per column, repeated wherever the system needs a different answer. A real desk needs several, so this one uses five — each from a different notebook, each doing the job it is good at.

| Part of the desk | Organisation | Control | Shape | From |
|---|---|---|---|---|
| inbound message → a specialist | hierarchical | the supervisor picks | one at a time | nb2 |
| support → engineering, directly | peer-to-peer | the agent that has the question | as needed | nb4 |
| what the case has established | blackboard | — (state, not control) | — | nb3 |
| risk recorded → the lawyer | blackboard | an event on the write | triggered | nb4 |
| "should the recall widen?" | isolated specialists | code, all at once | fan-out → `decisive` | nb3 |
| anything leaving the building | — | a gate in code | — | this notebook |

```
   message ──▶ supervisor ──asks──▶ support ──asks──▶ engineering
                   │                   │                  │
                   │                   └──── board ◀───────┘
                   │                           │
                   │                     risk written
                   │                           ▼
                   │                    event ──▶ lawyer ──▶ regulator_update
                   ▼
              reply (a form)
                   │
              sendable()? ──no──▶ held, with the reason
                   │
                  yes ──▶ out
```

The supervisor is hierarchical and support is peer-to-peer *at the same time*, which is the whole point: the columns are answered per edge, not per system.

### Making the writes happen

Look at the board after that run: `serial` and `in_recall`, nothing else. Engineering answered the question and never called `record`, so `cell_lot` and `risk` were never written — and because `risk` was never written, **the event never fired and the widening decision never ran**. Three of the five patterns were wired up and two of them sat there.

Notebook 3 found the same thing and said what to do about it: reading state is reliable because the code injects it, and writing should be reliable the same way. So the fields the board needs become **fields of the form**, and the code writes them:

```python
class Finding(BaseModel):     # engineering's version of submit
    answer: str
    cell_lot: str             # -> board, by the code
    risk: str                 # -> board, by the code, which fires the event
    confident: bool
    visibility: Literal["internal", "external"]
```

The agent still decides what goes in the fields. What it no longer has to do is remember a second tool call.

Now the board has a `risk`, so the condition in the next cell is true and the widening decision runs — the fifth pattern, which the first version of the desk skipped without saying so.

### The decision that needs more than one specialist

Routing sends a message to whoever should handle it. Some questions are not like that: *should the recall widen* has an engineering half, a legal half and a public half, and nobody owns it. So the code asks all three at once and decides with `decisive`, which notebook 3 found was the only rule that looked at what an agent had actually checked.

Whether to ask at all is a condition in code, reading the board — the cheapest control there is, and the one that never forgets.

## What the desk did, twice

**The first version worked for the customer and quietly skipped two of its own patterns.** The supervisor asked engineering, asked engineering again, then had support write the reply; support told the customer to stop charging and explained that the unit shares the recalled cell lot; the gate passed it (`confident: yes, visibility: external`). 13 calls, 21,632 tokens.

Then look at the board it left behind: `serial` and `in_recall`. No `cell_lot`, no `risk`. Engineering had called `cell_lot`, knew the answer, said it in its reply — and never called `record`. So `risk` was never written, so **the event never fired and the widening decision never ran**. Nothing failed. The system just did three of the five things it was built to do, and said nothing about the other two.

**The second version moves those fields into the form.** `Finding` has `cell_lot` and `risk` next to the answer, and the code writes them when the form comes back. The board ends complete, the event fires the lawyer, `regulator_update` gets written, and the condition on the board is true so the widening decision runs. 18 calls, 27,161 tokens: **26% more for the two patterns that had been silently dormant.**

The reply is also better, and for a reason worth naming: engineering's `risk` sentence was on the board while support wrote, so support's answer leads with "stop using them, stop charging" and then explains the cell lot. In the first version support was working from what the supervisor chose to pass on.

**One rough edge, left in.** The event fired **twice**, because the supervisor asked engineering twice and a watcher fires on every write. The lawyer therefore wrote two nearly identical regulator updates, and the second overwrote the first. Firing only on a *changed* value would not have helped — the two sentences differed slightly. The honest fixes are elsewhere: ask engineering once, or make the consumer idempotent, or have the lawyer read the field it is about to overwrite. A board plus events is not a queue, and it has no idea that two writes were about the same thing.

**And the decision the fan-out produced is defensible**, which is not the same as agreeing with it. All three specialists said not to widen the recall; engineering said so with its tools cited (`cell_lot` → VC-7731, `test_results` → 9 of 11, 2 undetermined, "low but not zero"), while the lawyer cited "none" and PR cited the brief. `majority` and `decisive` agree here, for entirely different reasons — one counted three votes, two of which had checked nothing, and the other read the only evidence in the room.

## Breaking it

Four ways this chapter has already broken, each reproduced deliberately. Three of them need no model at all: a stub stands in for the worker, which is the point — **a failure path you can only test by spending tokens is a failure path nobody tests.**

All three arrive as a `Response`, which is the whole point: the caller reads one shape whatever happened.

- **The worker that raised** comes back as *"Could not do this: engineer raised RuntimeError"*, with the original message in `unknowns`. Without this, the exception surfaces inside `Act`, which treats a failure as an argument problem and retries the whole worker through `DebugThink` — three model calls to repair arguments that were never wrong, and another agent run behind each.
- **The worker that ran out of budget** is the notebook 2 failure, caught. In notebook 2 `Runtime` ended a run over budget with an ordinary assistant message beginning *"Stopping: reached the maximum…"*, and the supervisor read that as an answer and carried on. `Runtime` now does better and worse at once: a stopped agent gets one last turn, with no tools, to say what its results established — so the text reads like a real answer, and nothing in it says the agent was cut off. What says so is `conversation.stopped`, which `messages.answer` turns into `unknowns`, and `attempt` into a failure form the gate refuses. The best-effort answer is kept in `unknowns`, because it is often most of the job.
- **The worker that would not use the form** is flagged by `messages.answer` itself: `confident: False`, and `unknowns` saying the agent answered as text so nothing in it is checked. Notice what this is *not* — nobody tried to parse the prose into fields. A form nobody filled in should not look like one that was.

Each of these is a stub standing in for a model. That matters more than it looks: **a failure path you can only exercise by spending tokens is a failure path nobody exercises.** Three of the four failures in this notebook cost nothing to test, so they can run on every change.

### The gate

`sendable` is four `if`s. Each one is a failure this chapter produced with a real model, and each would have been caught here:

Four `if`s, and every one of them is a real failure from earlier in the chapter:

- **internal, sent out** — notebook 1's one agent put the supplier's name and the recall cost into the regulator notice; notebook 2's supervisor forwarded a support answer to a journalist.
- **confident, nothing behind it** — notebook 2's *"no other batches are affected"*, which the database contradicts.
- **empty** — an agent that ran out of tokens mid-answer.
- **a failure** — anything `attempt` caught.

The order matters, and I got it wrong first: a failure is *also* internal and *also* not confident, so the check that fires first decides what the caller is told. Reporting "marked internal" for a dead API sends whoever reads the log looking at the wrong thing.

What the gate cannot do is notice that a confident, external, well-formed answer is **wrong**. `sendable` would have passed notebook 2's "for now, you're safe", because that reply was confident and meant for the customer. Everything the gate checks is a claim the agent made about its own answer. Catching the answer itself being wrong is what the board, the tools and the fan-out are for — and this chapter's evidence is that they help and do not settle it.

### Watchers that feed each other

The event in notebook 4 fires the lawyer when engineering records a risk. Nothing stops the lawyer's write from firing something that writes back.

The guard stops it at depth 3 and says what to look for: *"something is watching a field it also writes."* Two writes did happen before it refused, which is the honest outcome — a guard on the depth cannot undo what the chain already did, so the board is left part-way through and the caller has to deal with that.

This is the cost of events as a control. A manager picking speakers cannot loop like this, because something is choosing each time and can see the transcript. An event has no view of anything: it fires because a field was written, and if what it does writes that field again, it is a while-true. Keep chains one step deep unless you have a reason, and make the guard loud.

## What it cost

- **The desk costs about 21–27k tokens for one customer report**, 13–18 calls, on gpt-oss-120b. That is the honest price of five patterns: a supervisor call, two engineering runs, a support run, a lawyer run behind the event, and the forms in between.
- **The reliable version costs 26% more than the version that skipped two patterns.** Not because forms are expensive — they are the same call — but because the patterns that had been dormant actually ran.
- **The widening decision costs 5 calls** for three opinions and a verdict, and two of those opinions checked nothing. On this evidence it is worth asking engineering and treating the rest as advice.
- **The failure paths cost nothing.** Three stubs and a guard, no model. That is deliberate: they are the parts you want to run on every change.

## What this system is, and what it still gets wrong

Five patterns, one desk: a supervisor routing (hierarchical), support asking engineering itself (peer-to-peer), a board holding the case (blackboard), an event firing the lawyer (triggered), a fan-out deciding the question nobody owns — and one piece of plain code, the gate, in front of everything that leaves.

**What holds it together is not the agents.** It is the forms between them, the ownership on the board, the condition in code, and the four `if`s in `sendable`. The agents do the reading, the judging and the writing; every guarantee in the system is in the wiring.

**What it still gets wrong**, from its own outputs rather than from imagination:

- **It asked engineering twice** for one report, and fired the event twice, and the lawyer's second update overwrote its first. A board is not a queue and does not know two writes were about the same thing.
- **It cannot tell a wrong answer from a right one.** The gate reads what the agent claims about its answer, not the answer. Notebook 2's "for now, you're safe" would pass every check here.
- **`decisive` is only as good as the agent it names.** It refuses to decide when engineering is unconfident or checked nothing, which is right, and it has nothing to say about engineering being confidently mistaken.
- **Nothing here is measured.** Every live cell in this notebook is one run. Notebook 2 shows what it takes to claim more, and the numbers in notebooks 1–3 come from a different model.
- **Two boards, two model sizes, one scenario.** The chapter's findings are about gpt-oss-20b and 120b on one recall. The method transfers; the numbers do not.

## The chapter, in five lines

1. **A multi-agent system is several agents that are different on purpose** — different personas, different tools. Chapter 2's branches are the same agent again, and a fork is the right answer when one persona can do every part of a job too big for one context.
2. **A supervisor knows an agent by its description**, and routing you depend on belongs in the supervisor's own instructions: measured, 8 of 8 against 0 of 8.
3. **Shape the messages.** A reply with fields can be acted on; a typed request is refused before an agent runs. Self-reported fields are a place to look, enforced ones are a guarantee.
4. **Shared state says where information lives and never who runs next.** That is a separate choice: a manager, an event, or a fixed order.
5. **Failures have to arrive in the same shape as successes**, and the checks that keep a system honest are ordinary code — which is where a multi-agent system is usually thinnest, because the interesting part looks like it is happening inside the agents.